# Particion Temporal y Normalizacion con metodo por Transecto y metodo General

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 5 (original): Partición temporal y normalización.
Entrada: windows/ (generada por Código 4)
Salida: windows_partitioned/
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")
WINDOWS_DIR = os.path.join(BASE_DIR, "windows")
OUTPUT_DIR = os.path.join(BASE_DIR, "windows_partitioned")

INPUT_ML_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "ml")
INPUT_DL_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "dl")
INPUT_ML_GLOBAL = os.path.join(WINDOWS_DIR, "global", "ml")
INPUT_DL_GLOBAL = os.path.join(WINDOWS_DIR, "global", "dl")

OUTPUT_ML_TRANSECT = os.path.join(OUTPUT_DIR, "by_transect", "ml")
OUTPUT_DL_TRANSECT = os.path.join(OUTPUT_DIR, "by_transect", "dl")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_DIR, "global", "dl")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = 'O3'

# Fechas de corte
TRAIN_END = '2025-06-30 23:00:00'
VAL_START = '2025-07-01 00:00:00'
VAL_END = '2025-10-01 23:00:00'
TEST_START = '2025-10-02 00:00:00'

def load_entity_data(entity_dir, entity_name):
    X_path = os.path.join(entity_dir, f"{entity_name}_X.npy")
    y_path = os.path.join(entity_dir, f"{entity_name}_y.npy")
    ts_path = os.path.join(entity_dir, f"{entity_name}_timestamps.npy")
    if not (os.path.exists(X_path) and os.path.exists(y_path) and os.path.exists(ts_path)):
        return None, None, None
    X = np.load(X_path)
    y = np.load(y_path)
    timestamps = np.load(ts_path)
    timestamps = pd.to_datetime(timestamps)
    return X, y, timestamps

def split_by_timestamps(X, y, timestamps):
    train_mask = timestamps <= pd.to_datetime(TRAIN_END)
    val_mask = (timestamps >= pd.to_datetime(VAL_START)) & (timestamps <= pd.to_datetime(VAL_END))
    test_mask = timestamps >= pd.to_datetime(TEST_START)
    return (X[train_mask], y[train_mask]), (X[val_mask], y[val_mask]), (X[test_mask], y[test_mask])

def normalize_data(X_train, X_val, X_test, y_train, y_val, y_test):
    n_train, win_in, n_feat = X_train.shape
    n_val = X_val.shape[0]
    n_test = X_test.shape[0]

    X_train_flat = X_train.reshape(-1, n_feat)
    X_val_flat = X_val.reshape(-1, n_feat)
    X_test_flat = X_test.reshape(-1, n_feat)

    scaler_X = MinMaxScaler()
    X_train_scaled_flat = scaler_X.fit_transform(X_train_flat)
    X_val_scaled_flat = scaler_X.transform(X_val_flat)
    X_test_scaled_flat = scaler_X.transform(X_test_flat)

    X_train_sc = X_train_scaled_flat.reshape(n_train, win_in, n_feat)
    X_val_sc = X_val_scaled_flat.reshape(n_val, win_in, n_feat)
    X_test_sc = X_test_scaled_flat.reshape(n_test, win_in, n_feat)

    scaler_y = MinMaxScaler()
    y_train_flat = y_train.reshape(-1, 1)
    y_val_flat = y_val.reshape(-1, 1)
    y_test_flat = y_test.reshape(-1, 1)
    scaler_y.fit(y_train_flat)
    y_train_sc = scaler_y.transform(y_train_flat).reshape(y_train.shape)
    y_val_sc = scaler_y.transform(y_val_flat).reshape(y_val.shape)
    y_test_sc = scaler_y.transform(y_test_flat).reshape(y_test.shape)

    return X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y

def save_split_ml(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)

def save_split_dl(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test,
                  scaler_X, scaler_y):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)
    with open(os.path.join(save_dir, "scaler_X.pkl"), 'wb') as f:
        pickle.dump(scaler_X, f)
    with open(os.path.join(save_dir, "scaler_y.pkl"), 'wb') as f:
        pickle.dump(scaler_y, f)

def process_entity(entity_name, input_ml_dir, input_dl_dir, output_ml_dir, output_dl_dir):
    print(f"  Procesando {entity_name}...")
    X_ml, y_ml, ts_ml = load_entity_data(input_ml_dir, entity_name)
    if X_ml is None:
        return
    X_dl, y_dl, ts_dl = load_entity_data(input_dl_dir, entity_name)
    if X_dl is None:
        return
    if not np.array_equal(ts_ml, ts_dl):
        print(f"    Error: timestamps no coinciden. Se omite.")
        return

    (X_train_ml, y_train_ml), (X_val_ml, y_val_ml), (X_test_ml, y_test_ml) = split_by_timestamps(X_ml, y_ml, ts_ml)
    (X_train_dl, y_train_dl), (X_val_dl, y_val_dl), (X_test_dl, y_test_dl) = split_by_timestamps(X_dl, y_dl, ts_dl)

    save_split_ml(output_ml_dir, entity_name,
                  X_train_ml, y_train_ml, X_val_ml, y_val_ml, X_test_ml, y_test_ml)

    if len(X_train_dl) > 0:
        X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y = normalize_data(
            X_train_dl, X_val_dl, X_test_dl, y_train_dl, y_val_dl, y_test_dl
        )
        save_split_dl(output_dl_dir, entity_name,
                      X_train_sc, y_train_sc, X_val_sc, y_val_sc, X_test_sc, y_test_sc,
                      scaler_X, scaler_y)
    else:
        print(f"    No hay datos de entrenamiento para {entity_name}, se omite DL.")

def process_by_transect():
    print("\n--- Procesando datos por transecto ---")
    if not os.path.exists(INPUT_ML_TRANSECT) or not os.path.exists(INPUT_DL_TRANSECT):
        return
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_TRANSECT).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_TRANSECT).glob("*_X.npy")]
    common = set(ml_files) & set(dl_files)
    for entity in sorted(common):
        process_entity(entity, INPUT_ML_TRANSECT, INPUT_DL_TRANSECT,
                       OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT)

def process_global():
    print("\n--- Procesando datos globales (por estación) ---")
    if not os.path.exists(INPUT_ML_GLOBAL) or not os.path.exists(INPUT_DL_GLOBAL):
        return
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_GLOBAL).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_GLOBAL).glob("*_X.npy")]
    common = set(ml_files) & set(dl_files)
    for entity in sorted(common):
        process_entity(entity, INPUT_ML_GLOBAL, INPUT_DL_GLOBAL,
                       OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL)

if __name__ == "__main__":
    print("="*60)
    print("Partición temporal y normalización de ventanas")
    print("="*60)
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")

Partición temporal y normalización de ventanas

--- Procesando datos por transecto ---
  Procesando Transecto_1...
  Procesando Transecto_2...

--- Procesando datos globales (por estación) ---
  Procesando T1_E1_Alicante...
  Procesando T1_E2_Elda...
  Procesando T2_E1_Elche...
  Procesando T2_E2_Elda...

Proceso completado. Revise la carpeta:
  /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/windows_partitioned
